# 폴더 정리 후 경로·무결성 검증

2026-10-06 사용자 요청에 따라 코드·데이터·읽는 보고서·계산 결과를 분리했다. 원래 파일과 명세는 ZIP으로 보존했다. 아래 함수는 경로 정리 이후 명시적 파일 계약의 해시를 갱신하는 관리용 함수이며 분석 조건·원자료는 변경하지 않는다. 함수를 정의할 뿐 자동으로 갱신하지 않는다.

In [1]:
from pathlib import Path
import json,hashlib
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'metadata/organization/20261006_moves.json').is_file())
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
MOVES=json.loads((ROOT/'metadata/organization/20261006_moves.json').read_text(encoding='utf-8'))['moves']
MAPPING={r['old_path']:r['new_path'] for r in MOVES}


In [2]:
def refresh_contract(relative, reason):
    path=ROOT/relative;j=json.loads(path.read_text(encoding='utf-8-sig'));updates=[]
    def one(node):
        if isinstance(node,list):
            for value in node:one(value)
        if not isinstance(node,dict):return
        # Explicit file records only. Raw/data contents must still match the previous digest.
        if 'path' in node and 'sha256' in node:
            old=node['path'];new=MAPPING.get(old,old);p=ROOT/new
            if p.is_file():
                digest=sha(p)
                if new.startswith('data/') and digest!=node['sha256']:raise AssertionError('Data hash changed: '+new)
                if old!=new or node['sha256']!=digest:updates.append({'path':new,'old_sha256':node['sha256'],'new_sha256':digest})
                node['path']=new;node['sha256']=digest
                if 'bytes' in node:node['bytes']=p.stat().st_size
        for key in ['protected_source_sha256','protected_sha256','source_sha256','output_sha256']:
            if key in node and isinstance(node[key],dict):
                result={}
                for old,digest in node[key].items():
                    new=MAPPING.get(old,old);p=ROOT/new
                    if not p.is_file():p=path.parent/new
                    current=sha(p) if p.is_file() else digest
                    if new.startswith('data/') and current!=digest:
                        check=ROOT/'metadata/organization/20261006_rdf_provenance_check.json'
                        allowed=new=='data/analysis/final/knowledge_graph/knowledge_graph.ttl' and check.exists() and json.loads(check.read_text(encoding='utf-8')).get('only_source_provenance_changed') is True
                        if not allowed:raise AssertionError('Data hash changed: '+new)
                    if old!=new or digest!=current:updates.append({'path':new,'old_sha256':digest,'new_sha256':current})
                    result[new]=current
                node[key]=result
        for value in node.values():one(value)
    one(j)
    if updates:
        j['organization_update']={'date':'2026-10-06','reason':reason,'original_snapshot':'archive/20261006_before_organization.zip','analytical_specification_changed':False}
        path.write_text(json.dumps(j,ensure_ascii=False,indent=2),encoding='utf-8')
    return updates


## 이동 후 확인

모든 이동 대상이 남아 있는지, 보고서에 계산용 파일이 섞이지 않았는지, 안내 문서의 링크와 핵심 통계 결과가 유지되는지 확인한다. 새 커널 실행은 수정한 운영 노트북별 실행 기록에서 확인한다.

In [3]:
import re,io,zipfile
import pandas as pd
from urllib.parse import unquote
from IPython.display import display
checks=[]
checks.append({'check':'All moved files present','passed':all((ROOT/r['new_path']).is_file() for r in MOVES),'detail':len(MOVES)})
bad_reports=[p.relative_to(ROOT).as_posix() for p in (ROOT/'reports').rglob('*') if p.is_file() and p.suffix.lower() not in ['.html','.md','.pdf']]
checks.append({'check':'Reports contain readable documents only','passed':not bad_reports,'detail':len(bad_reports)})
checks.append({'check':'Old mixed final/test folders removed','passed':not (ROOT/'final').exists() and not (ROOT/'test').exists(),'detail':0})
links=[]
for p in [ROOT/'notebooks/main/README.md',ROOT/'notebooks/experiments/README.md',ROOT/'results/main/baseline/README.md',ROOT/'results/experiments/README.md']:
 for value in re.findall(r'\]\(([^)]+)\)',p.read_text(encoding='utf-8')):
  if ':' in value or value.startswith('#'):continue
  target=(p.parent/unquote(value.split('#')[0])).resolve();links.append({'document':p.relative_to(ROOT).as_posix(),'link':value,'exists':target.exists()})
checks.append({'check':'Guide links resolve','passed':all(r['exists'] for r in links),'detail':len(links)})


In [4]:
before=ROOT/'archive/20261006_before_organization.zip'
core=['reports/final/09_industry_effects.csv','reports/final/10_gee_industry_effects.csv','reports/final/time_windows/11_industry_effects.csv','reports/final/revised_quarterly/13_effects.csv',
      'test/reports/01_specification_results.csv','test/reports/02_results.csv','test/reports/03_power.csv','test/reports/05_results.csv','test/reports/07_results.csv']
comparisons=[]
with zipfile.ZipFile(before) as z:
 for old in core:
  a=pd.read_csv(io.BytesIO(z.read(old)));b=pd.read_csv(ROOT/MAPPING[old]);numeric=list(a.select_dtypes(include='number').columns)
  pd.testing.assert_frame_equal(a[numeric],b[numeric],check_exact=False,rtol=1e-10,atol=1e-12)
  comparisons.append({'original':old,'current':MAPPING[old],'rows':len(b),'numeric_columns':len(numeric),'unchanged':True})
checks.append({'check':'Key numerical result tables unchanged','passed':all(r['unchanged'] for r in comparisons),'detail':len(comparisons)})
result={'status':'passed' if all(r['passed'] for r in checks) else 'failed','date':'2026-10-06','checks':checks,'statistical_comparisons':comparisons,'guide_links':links,'reports_non_documents':bad_reports}
(ROOT/'metadata/organization/20261006_validation.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(checks));assert result['status']=='passed'


,check,passed,detail
0,All moved files present,True,651
1,Reports contain readable documents only,True,0
2,Old mixed final/test folders removed,True,0
3,Guide links resolve,True,134
4,Key numerical result tables unchanged,True,9


## 해석

통과는 저장 위치 변경 후 파일과 핵심 통계 수치가 유지됐음을 뜻한다. 이전 실행의 출처·해시 기록은 정리 전 ZIP에 보존하며, 새 실행 기록을 이전 실행의 기록으로 소급하지 않는다.